# Libraries

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.ensemble import StackingRegressor

import pandas as pd
import ast, re
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Data Loading

In this step, all datasets used in the analysis are loaded into the notebook. These include the international job postings dataset, the **Philippine Labor Force Survey (LFS)** dataset, and **O*NET reference files** containing *occupation titles and skills*. The shapes of the datasets are printed to confirm successful loading, and a preview of the job postings dataset is displayed for initial inspection.

In [2]:
# International job postings
jobs = pd.read_csv("data_jobs.csv")

# Local LFS data
lfs = pd.read_csv("FIES2015 - LFSJAN16 CSV - Cleaned.csv", low_memory=False)

# World Bank economic data
econ = pd.read_csv("world_bank_data_2025.csv")

# Occupation reference files
# occupation = pd.read_excel("db_30_2_excel/Occupation Data.xlsx")
# titles = pd.read_excel("db_30_2_excel/Alternate Titles.xlsx")
# skills = pd.read_excel("db_30_2_excel/Skills.xlsx")
# tech_skills = pd.read_excel("db_30_2_excel/Technology Skills.xlsx")
# reported_titles = pd.read_excel("db_30_2_excel/Sample of Reported Titles.xlsx")

In [3]:
econ_2023 = econ[econ["year"] == 2023].copy()
econ_2023 = econ_2023[[
    "country_name",
    "GDP per Capita (Current USD)",
    "GDP (Current USD)",
    "Unemployment Rate (%)",
    "Inflation (CPI %)",
    "GDP Growth (% Annual)"
]]
econ_2023.columns = [
    "country",
    "gdp_per_capita",
    "gdp",
    "unemployment",
    "inflation",
    "gdp_growth"
]

# Preprocessing (International Dataset)

In [4]:
country_map = {
    "Russia": "Russian Federation",
    "Vietnam": "Viet Nam",
    "South Korea": "Korea, Rep.",
    "North Korea": "Korea, Dem. People's Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Iran": "Iran, Islamic Rep.",
    "Venezuela": "Venezuela, RB",
    "Slovakia": "Slovak Republic",
    "Hong Kong": "Hong Kong SAR, China",
    "Macedonia (FYROM)": "North Macedonia",
    "Palestine": "West Bank and Gaza",
    "Laos": "Lao PDR",
    "Syria": "Syrian Arab Republic",
    "Turkey": "Turkiye",
    "Yemen": "Yemen, Rep.",
    "United States Virgin Islands": "Virgin Islands (U.S.)",
}

country_map.update({
    "Bahamas": "Bahamas, The",
    "Brunei": "Brunei Darussalam",
    "Congo, Democratic Republic of the": "Congo, Dem. Rep.",
    "Curaçao": "Curacao",
    "Gambia": "Gambia, The",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Taiwan": "Taiwan, China",
    "U.S. Virgin Islands": "Virgin Islands (U.S.)",
})

In [5]:
jobs_model = jobs[[
    "job_title_short", "job_country",
    "salary_rate", "salary_year_avg", "salary_hour_avg",
    "job_skills"
]].copy()

jobs_model["job_country"] = jobs_model["job_country"].replace(country_map)

jobs_model = jobs_model.merge(
    econ_2023,
    left_on="job_country",
    right_on="country",
    how="left"
)

In [6]:
for col in ["gdp_per_capita","gdp","unemployment","inflation","gdp_growth"]:
    jobs_model[col] = jobs_model[col].fillna(jobs_model[col].median())

In [7]:
def get_salary(row):
    if pd.notna(row["salary_year_avg"]):
        return row["salary_year_avg"]
    if pd.notna(row["salary_hour_avg"]):
        return row["salary_hour_avg"] * 40 * 52
    return None

jobs_model["salary"] = jobs_model.apply(get_salary, axis=1)
jobs_model = jobs_model.dropna(subset=["salary"])

In [8]:
# Remove salary outliers (IQR method)
Q1 = jobs_model["salary"].quantile(0.25)
Q3 = jobs_model["salary"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

jobs_model = jobs_model[
    (jobs_model["salary"] >= lower) & 
    (jobs_model["salary"] <= upper)
]

In [9]:
# Log transforms
jobs_model["gdp_per_capita_log"] = np.log1p(jobs_model["gdp_per_capita"])
jobs_model["gdp_log"] = np.log1p(jobs_model["gdp"])

# Labor market strength
jobs_model["labor_market_tightness"] = 1 - jobs_model["unemployment"]

# Real (inflation-adjusted)
jobs_model["real_gdp_per_capita"] = (
    jobs_model["gdp_per_capita"] / (1 + jobs_model["inflation"]/100)
)

## Preprocessing the Job Roles

*Note:* No filtering for data industry roles was required because the dataset already consists of job postings related to data and analytics roles. According to the dataset description, it contains real-world data analytics job postings collected from various online job sources. Therefore, all job titles are assumed to belong to the data industry. A count of job titles was performed to examine the distribution of roles in the dataset.

**Source**
- https://huggingface.co/datasets/lukebarousse/data_jobs

In [10]:
jobs_model["job_title_short"].value_counts()

job_title_short
Data Analyst                 9566
Data Scientist               8289
Data Engineer                6681
Senior Data Engineer         1980
Senior Data Scientist        1942
Senior Data Analyst          1476
Business Analyst              996
Machine Learning Engineer     610
Software Engineer             561
Cloud Engineer                 85
Name: count, dtype: int64

In [11]:
title_counts = jobs_model["job_title_short"].value_counts()
common_titles = title_counts[title_counts >= 100].index
jobs_model = jobs_model[jobs_model["job_title_short"].isin(common_titles)]

This **function** cleans and standardizes the skill data by converting different formats (lists, strings, or missing values) into a consistent list of individual skills. It removes extra characters, splits merged skills, and ensures each job posting returns a clean list of skills.

In [12]:
def clean_skills(x):
    if isinstance(x, list):
        all_items = []
        for item in x:
            text = str(item).strip().strip("'\"[]")
            parts = re.split(r",\s*", text)
            all_items.extend(parts)
        return [s.strip().strip("'\" ").lower() for s in all_items if s.strip()]
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return clean_skills(parsed)
        except:
            pass
        x_clean = x.strip().strip("[]")
        parts = re.split(r",\s*", x_clean)
        return [s.strip().strip("'\" ").lower() for s in parts if s.strip()]
    return []

Applying the `clean_skills` function to the `job_skills` column to create a clean and standardized list of skills for each job posting.

In [13]:
jobs_model["skills_list"] = jobs_model["job_skills"].apply(clean_skills)

This step removes duplicate skills within each row while preserving the original order of skills.

In [14]:
jobs_model["skills_list"] = jobs_model["skills_list"].apply(lambda x: list(dict.fromkeys(x)))

In [15]:
jobs_model["skills_list"] = jobs_model["skills_list"].apply(lambda x: x if isinstance(x, list) else [])

This step ensures that all values in `skills_list` are valid lists to avoid errors in later processing.

In [16]:
jobs_model["skill_count"] = jobs_model["skills_list"].apply(len)

This dictionary maps countries to broader geographic regions (e.g., NA, EU, SEA). It allows the dataset to group job postings by region for easier comparison and analysis.

In [17]:
region_map = {

    # North America
    "United States": "NA",
    "Canada": "NA",

    # Latin America
    "Mexico": "LATAM",
    "Costa Rica": "LATAM",
    "Honduras": "LATAM",
    "Guatemala": "LATAM",
    "Panama": "LATAM",
    "Bahamas": "LATAM",
    "Jamaica": "LATAM",
    "Puerto Rico": "LATAM",
    "Dominican Republic": "LATAM",
    "Colombia": "LATAM",
    "Brazil": "LATAM",
    "Argentina": "LATAM",
    "Chile": "LATAM",
    "Peru": "LATAM",
    "Uruguay": "LATAM",
    "Bolivia": "LATAM",
    "Paraguay": "LATAM",
    "Venezuela": "LATAM",
    "El Salvador": "LATAM",
    "Nicaragua": "LATAM",

    # Europe
    "United Kingdom": "EU",
    "Germany": "EU",
    "France": "EU",
    "Spain": "EU",
    "Belgium": "EU",
    "Denmark": "EU",
    "Hungary": "EU",
    "Poland": "EU",
    "Portugal": "EU",
    "Sweden": "EU",
    "Estonia": "EU",
    "Croatia": "EU",
    "Netherlands": "EU",
    "Ireland": "EU",
    "Lithuania": "EU",
    "Slovenia": "EU",
    "Austria": "EU",
    "Greece": "EU",
    "Czechia": "EU",
    "Switzerland": "EU",
    "Italy": "EU",
    "Norway": "EU",
    "Luxembourg": "EU",
    "Romania": "EU",
    "Finland": "EU",
    "Latvia": "EU",
    "Slovakia": "EU",
    "Serbia": "EU",
    "Montenegro": "EU",
    "Bosnia and Herzegovina": "EU",
    "Belarus": "EU",
    "Ukraine": "EU",

    # Southeast Asia
    "Philippines": "SEA",
    "Singapore": "SEA",
    "Indonesia": "SEA",
    "Malaysia": "SEA",
    "Thailand": "SEA",
    "Vietnam": "SEA",
    "Cambodia": "SEA",
    "Myanmar": "SEA",
    "Brunei": "SEA",

    # East Asia
    "Japan": "EAST_ASIA",
    "South Korea": "EAST_ASIA",
    "Taiwan": "EAST_ASIA",
    "China": "EAST_ASIA",
    "Hong Kong": "EAST_ASIA",

    # South Asia
    "India": "SOUTH_ASIA",
    "Pakistan": "SOUTH_ASIA",
    "Bangladesh": "SOUTH_ASIA",
    "Sri Lanka": "SOUTH_ASIA",
    "Nepal": "SOUTH_ASIA",

    # Middle East
    "Israel": "ME",
    "United Arab Emirates": "ME",
    "Jordan": "ME",
    "Turkey": "ME",
    "Lebanon": "ME",

    # Africa
    "South Africa": "AFRICA",
    "Nigeria": "AFRICA",
    "Kenya": "AFRICA",
    "Ghana": "AFRICA",
    "Morocco": "AFRICA",
    "Egypt": "AFRICA",
    "Zimbabwe": "AFRICA",
    "Namibia": "AFRICA",
    "Tunisia": "AFRICA",
    "Algeria": "AFRICA",
    "Uganda": "AFRICA",
    "Cameroon": "AFRICA",
    "Senegal": "AFRICA",
    "Mali": "AFRICA",
    "Benin": "AFRICA",
    "Sudan": "AFRICA",
    "Zambia": "AFRICA",

    # Oceania
    "Australia": "OCEANIA",
    "New Zealand": "OCEANIA"
}

In [18]:
jobs_model["region"] = jobs_model["job_country"].map(region_map).fillna("OTHER")

In [19]:
mlb = MultiLabelBinarizer()
skills_matrix = mlb.fit_transform(jobs_model["skills_list"])
skills_df = pd.DataFrame(skills_matrix, columns=mlb.classes_, index=jobs_model.index)

In [20]:
skill_counts = skills_df.sum().sort_values(ascending=False)
top_skills = skill_counts.head(150).index
skills_df = skills_df[top_skills]

In [21]:
# Drop old skill columns
jobs_model = jobs_model.drop(columns=mlb.classes_, errors="ignore")

# Model Training

In [22]:
jobs_model = pd.concat([jobs_model, skills_df], axis=1)

In [23]:
region_dummies = pd.get_dummies(jobs_model["region"], prefix="region")
title_dummies = pd.get_dummies(jobs_model["job_title_short"], prefix="title")
jobs_model = pd.concat([jobs_model, region_dummies, title_dummies], axis=1)

In [24]:
jobs_model["cloud_skills"] = jobs_model[["aws","azure","gcp"]].sum(axis=1)
jobs_model["bi_tools"] = jobs_model[["tableau","power bi","looker","qlik"]].sum(axis=1)
jobs_model["ml_tools"] = jobs_model[["tensorflow","pytorch","scikit-learn","keras"]].sum(axis=1)
jobs_model["is_US"] = (jobs_model["job_country"] == "United States").astype(int)
jobs_model["salary_log"] = np.log1p(jobs_model["salary"])

In [25]:
X = jobs_model.drop(columns=[
    "job_title_short","job_country","country",  # add this
    "salary_rate","salary_year_avg","salary_hour_avg",
    "job_skills","skills_list","region",
    "salary","salary_log"
])

In [26]:
y = jobs_model["salary_log"]
text_data = jobs_model["job_skills"].fillna("").astype(str)

In [27]:
X_train, X_test, y_train, y_test, text_train, text_test = train_test_split(
    X, y, text_data, test_size=0.2, random_state=42
)

In [28]:
tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_text_train = tfidf.fit_transform(text_train)
X_text_test = tfidf.transform(text_test)

In [29]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# convert structured features to sparse format
X_train_structured_sparse = csr_matrix(X_train_scaled)
X_test_structured_sparse = csr_matrix(X_test_scaled)

# combine structured + TF-IDF text features
X_train_combined = hstack([X_train_structured_sparse, X_text_train])
X_test_combined = hstack([X_test_structured_sparse, X_text_test])

In [30]:
lr = LinearRegression()
rf = RandomForestRegressor(random_state=42)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=5, random_state=42)
xgb = XGBRegressor(n_estimators=700, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)

In [32]:
# param_grid = {"n_estimators": [300,500,700], "max_depth":[10,15,20,None], "min_samples_split":[5,10,20]}
# search = RandomizedSearchCV(rf, param_distributions=param_grid, n_iter=10, cv=3, scoring="r2", n_jobs=-1, random_state=42)
# search.fit(X_train, y_train)
# rf_best = search.best_estimator_
# print("Best RF parameters:", search.best_params_)

rf_best = RandomForestRegressor(
    n_estimators=700,
    min_samples_split=20,
    max_depth=None,
    random_state=42
)

In [33]:
stack = StackingRegressor(
    estimators=[
        ('rf', rf_best),
        ('xgb', xgb)
    ],
    final_estimator=LinearRegression(),
    n_jobs=-1
)

In [ ]:
lr.fit(X_train_combined, y_train)
rf_best.fit(X_train, y_train)   # structured only
gb.fit(X_train, y_train)        # structured only
xgb.fit(X_train_combined, y_train)
stack.fit(X_train_combined, y_train)

print("All models trained successfully!")

In [ ]:
# Predictions in log space
lr_pred_log = lr.predict(X_test_combined)
rf_pred_log = rf_best.predict(X_test)
gb_pred_log = gb.predict(X_test)
xgb_pred_log = xgb.predict(X_test_combined)
stack_pred_log = stack.predict(X_test_combined)

# Convert to real salary
lr_pred = np.expm1(lr_pred_log)
rf_pred = np.expm1(rf_pred_log)
gb_pred = np.expm1(gb_pred_log)
xgb_pred = np.expm1(xgb_pred_log)
stack_pred = np.expm1(stack_pred_log)

# Actual values
y_test_salary = np.expm1(y_test)

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression","Random Forest","Gradient Boosting","XGBoost","Stacking"],
    "R2": [
        r2_score(y_test, lr_pred_log),
        r2_score(y_test, rf_pred_log),
        r2_score(y_test, gb_pred_log),
        r2_score(y_test, xgb_pred_log),
        r2_score(y_test, stack_pred_log)
    ],
    "MAE": [
        mean_absolute_error(y_test_salary, lr_pred),
        mean_absolute_error(y_test_salary, rf_pred),
        mean_absolute_error(y_test_salary, gb_pred),
        mean_absolute_error(y_test_salary, xgb_pred),
        mean_absolute_error(y_test_salary, stack_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test_salary, lr_pred)),
        np.sqrt(mean_squared_error(y_test_salary, rf_pred)),
        np.sqrt(mean_squared_error(y_test_salary, gb_pred)),
        np.sqrt(mean_squared_error(y_test_salary, xgb_pred)),
        np.sqrt(mean_squared_error(y_test_salary, stack_pred))
    ]
})

               Model        R2           MAE          RMSE
0  Linear Regression  0.266681  29242.808447  37536.654302
1      Random Forest  0.311012  27859.474236  36219.229425
2  Gradient Boosting  0.286746  28961.018754  37105.575556
3            XGBoost  0.304236  28470.148243  36599.178104


In [ ]:
cv_scores = cross_val_score(rf_best, X, y, cv=5, scoring="r2")
print("RF CV mean R2:", cv_scores.mean())

KeyboardInterrupt: 